In [1]:
import pandas as pd

In [51]:
xlsx = pd.ExcelFile("Unifesp_Demanda_enriched.xlsx")

orders_df = pd.read_excel(xlsx, sheet_name="Demanda")
schedule_df = pd.read_excel(xlsx, sheet_name="Calendário")
holidays_df = pd.read_excel(xlsx, sheet_name="Feriados")

In [ ]:
holidays_df["feriado"] = holidays_df["tipo"].eq("Feriado Nacional")

orders_df = orders_df.merge(
    holidays_df[["data", "feriado"]],
    how="left",
    left_on="data_pedido",
    right_on="data"
).drop(columns={"data"})

orders_df["feriado"] = orders_df["feriado"].fillna(False)

## Adicionando variáveis de tempo na base

In [60]:
orders_df["data_pedido"] = pd.to_datetime(
    orders_df["data_pedido"],
    errors="coerce"
)

schedule_df["Dt Abertura"] = pd.to_datetime(
    schedule_df["Dt Abertura"],
    errors="coerce"
)

schedule_df["Dt Fechamento"] = pd.to_datetime(
    schedule_df["Dt Fechamento"],
    errors="coerce"
)


orders_df["ano"] = orders_df["data_pedido"].dt.year
orders_df["mes"] = orders_df["data_pedido"].dt.month
orders_df["dia_mes"] = orders_df["data_pedido"].dt.day
orders_df["dia_semana_num"] = orders_df["data_pedido"].dt.dayofweek
orders_df["semana_ano"] = orders_df["data_pedido"].dt.isocalendar().week.astype("Int64")
orders_df["trimestre"] = orders_df["data_pedido"].dt.quarter

day_names = {
    0: "Segunda",
    1: "Terca",
    2: "Quarta",
    3: "Quinta",
    4: "Sexta",
    5: "Sabado",
    6: "Domingo"
}

orders_df["dia_semana"] = (
    orders_df["dia_semana_num"]
    .map(day_names)
)

In [61]:
orders_df["nm_ciclo"] = (
    orders_df["nm_ciclo"]
    .astype(str)
    .str.zfill(2)
)

orders_df["CICLOS"] = (
    orders_df["aa_ciclo"].astype(str)
    + orders_df["nm_ciclo"]
)

schedule_df = schedule_df.rename(columns={"COD SETOR": "cd_setor"})

schedule_df["cd_setor"] = schedule_df["cd_setor"].astype(str)
schedule_df["CICLOS"] = schedule_df["CICLOS"].astype(str)

orders_df["cd_setor"] = orders_df["cd_setor"].astype(str).str[1:]

orders_schedule_df = orders_df.merge(
    schedule_df,
    on=["cd_setor", "CICLOS"],
    how="left"
)

orders_schedule_df["IsDateValid"] = (
    orders_schedule_df["data_pedido"]
    .between(
        orders_schedule_df["Dt Abertura"],
        orders_schedule_df["Dt Fechamento"],
        inclusive="left"
    )
)

orders_schedule_df = orders_schedule_df[orders_schedule_df["IsDateValid"]==True]


In [62]:
orders_schedule_df["dia_ciclo"] = (
    orders_schedule_df["data_pedido"] - orders_schedule_df["Dt Abertura"]
).dt.days + 1

orders_schedule_df["dias_para_fechamento"] = (
    orders_schedule_df["Dt Fechamento"] - orders_schedule_df["data_pedido"]
).dt.days

orders_df = orders_df.sort_values(
    ["cd_setor", "CICLOS", "data_pedido"]
)

orders_schedule_df["dia_captacao"] = (
    orders_schedule_df
    .groupby(["cd_setor", "CICLOS"])["data_pedido"]
    .rank(method="dense")
    .astype("Int64")
)

In [64]:
orders_schedule_df.to_csv("base_tratada.csv", index = False)

In [4]:
orders_schedule_before_2026_df = orders_schedule_df[
    orders_schedule_df["data_pedido"].dt.year < 2026
].copy()

orders_schedule_2026_df= orders_schedule_df[
    orders_schedule_df["data_pedido"].dt.year == 2026
].copy()

In [31]:
orders_schedule_2026_df.shape

(138902, 21)

In [30]:
orders_schedule_2026_df[
    orders_schedule_2026_df["Dt Abertura"].isna()
].copy()

,data_pedido,cd_cd,cd_setor,rota,cidade,estado,nm_ciclo,aa_ciclo,total_pedidos_mascarado,total_volumes_mascarado,...,CICLOS,BLOCO,SUB BLOCO,Dt Abertura,Dt Fechamento,Dt Estatistica,#,Qtde dias,IsDateValid,IsToRemove
437,2026-07-09,5800,3940,138700.0,MOGI GUACU,SP,11,2026,28,36,...,202611,NaN,NaN,NaT,NaT,NaT,NaN,NaN,False,True
510,2026-07-25,5600,3943,133700.0,BRASILIA,DF,12,2026,31,111,...,202612,NaN,NaN,NaT,NaT,NaT,NaN,NaN,False,True
542,2026-02-18,5100,3950,124750.0,RECIFE,PE,03,2026,28,47,...,202603,NaN,NaN,NaT,NaT,NaT,NaN,NaN,False,True
573,2026-01-07,5600,3301,8809.0,ITUMBIARA,GO,19,2025,28,28,...,202519,NaN,NaN,NaT,NaT,NaT,NaN,NaN,False,True
801,2026-06-08,5700,4000,122072.0,POA,SP,09,2026,28,37,...,202609,NaN,NaN,NaT,NaT,NaT,NaN,NaN,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399361,2026-08-22,5800,3940,138700.0,SALTO,SP,13,2026,28,48,...,202613,NaN,NaN,NaT,NaT,NaT,NaN,NaN,False,True
399395,2026-03-13,5700,0999,730206.0,CAMPINAS,SP,05,2026,28,69,...,202605,NaN,NaN,NaT,NaT,NaT,NaN,NaN,False,True
399480,2026-04-25,5100,3950,125621.0,FORTALEZA,CE,07,2026,29,68,...,202607,NaN,NaN,NaT,NaT,NaT,NaN,NaN,False,True
399806,2026-01-08,5700,3414,425.0,DOMINGOS MARTINS,ES,19,2025,28,34,...,202519,NaN,NaN,NaT,NaT,NaT,NaN,NaN,False,True


In [33]:
(
    orders_schedule_2026_df.loc[
        orders_schedule_2026_df["Dt Abertura"].isna(),
        ["cd_setor", "CICLOS"]
    ]
    .drop_duplicates()
    .shape[0]
)

475

In [35]:
(
    orders_schedule_2026_df.loc[
        orders_schedule_2026_df["Dt Abertura"].isna(),
        ["cd_setor"]
    ]
    .drop_duplicates()
    .shape[0]
)

231

In [ ]:
#Número total de setores
orders_schedule_2026_df["cd_setor"].drop_duplicates().shape[0]

676